# Instructions

### Notebook needs to be migrated to Gemini.

# Workshop: Multi-Agent LLMs for Finance

In this example, we demonstrate a multi-agent system using large language models (LLMs) applied to finance. The system is composed of three agents:

1. **Data Agent:** Fetches financial data (e.g., historical stock prices for a ticker such as AAPL) using the `yfinance` library.
2. **Analysis Agent:** Reviews the data by computing basic statistics and provides a neutral analysis using an LLM (or simulated response).
3. **Decision Agent:** Generates a trading recommendation (Buy, Hold, or Sell) based on the analysis.

This notebook includes both code and explanatory text. You may need to set your OpenAI API key if you wish to use real API calls.


In [4]:
# Import necessary libraries
import yfinance as yf
import pandas as pd
import datetime
# import genai

import os

from google.genai import Client

# Set your OpenAI API key if you have one; otherwise, the agents will return simulated responses.
# Uncomment and set your API key below:


In [6]:
MODEL_ID = 'gemini-3-flash-preview'
MODEL_ID = 'gemini-3.1-flash-lite-preview'
MODEL_ID = 'gemini-2.5-flash-lite'

PROJECT_ID = 'rogue-trader-friday'

LOCATION = 'global'

client = Client(vertexai=True, project=PROJECT_ID, location=LOCATION)



## Define Agents

### Define Agent Base **Class**

In [19]:
# Define a base agent class that uses the OpenAI API if an API key is set; otherwise, it returns a simulated response.
class BaseAgent:
    def __init__(self, name, role, client):
        self.name = name
        self.role = role  # This describes the agent's responsibility
        self.client = client

    def act(self, prompt):
        response = client.models.generate_content(
            model=MODEL_ID, contents=prompt, 
        )
        return response.text


### Define Data Agent

In [20]:
# Data Agent: Fetches financial data using yfinance.
class DataAgent(BaseAgent):
    def fetch_data(self, ticker, period='1mo', interval='1d'):
        print(f"{self.name} is fetching data for {ticker} (Period: {period}, Interval: {interval}).")
        stock = yf.Ticker(ticker)
        df = stock.history(period=period, interval=interval)
        return df



### Define Analysis Agent

In [21]:
# Analysis Agent: Analyzes the data by computing summary statistics and generating insights.
class AnalysisAgent(BaseAgent):
    def analyze(self, data_df):
        # Compute summary statistics from the DataFrame.
        summary = data_df.describe().to_string()
        prompt = f"Analyze the following financial data summary and provide neutral insights:\n{summary}"
        return self.act(prompt)

### Define Decision Agent

In [22]:
# Decision Agent: Provides a trading recommendation based on the analysis.
class DecisionAgent(BaseAgent):
    def recommend(self, analysis):
        prompt = f"Based on the following analysis, provide a simple trading recommendation (Buy, Hold, or Sell) with reasoning:\n{analysis}"
        return self.act(prompt)


## Simulate Agent Interaction

In [23]:
# Create instances of the agents with specific roles.
data_agent = DataAgent("DataAgent", "You fetch and provide historical stock data.", client)
analysis_agent = AnalysisAgent("AnalysisAgent", "You analyze financial data and provide neutral insights.", client)
decision_agent = DecisionAgent("DecisionAgent", "You provide trading recommendations based on analysis.", client)

# Step 1: Data Agent fetches historical data for a chosen stock.
ticker = "AAPL"  # Example ticker for Apple Inc.
data_df = data_agent.fetch_data(ticker, period='12mo', interval='1d')
print("\nFetched Data (last 5 records):")
print(data_df.tail())



DataAgent is fetching data for AAPL (Period: 12mo, Interval: 1d).

Fetched Data (last 5 records):
                                 Open        High         Low       Close  \
Date                                                                        
2026-04-08 00:00:00-04:00  258.450012  259.750000  256.529999  258.899994   
2026-04-09 00:00:00-04:00  259.000000  261.119995  256.070007  260.489990   
2026-04-10 00:00:00-04:00  259.980011  262.190002  259.019989  260.480011   
2026-04-13 00:00:00-04:00  259.730011  260.179993  256.660004  259.200012   
2026-04-14 00:00:00-04:00  259.250000  261.929993  257.190002  258.829987   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-04-08 00:00:00-04:00  41032800        0.0           0.0  
2026-04-09 00:00:00-04:00  28121600        0.0           0.0  
2026-04-10 00:00:00-04:00  31291500        0.0           0.0  
2026-04-13 00:00:00-04:00  36234700        0.0 

In [24]:
# Step 2: Analysis Agent analyzes the fetched data.
analysis = analysis_agent.analyze(data_df)
print("\nAnalysis:")
print(analysis)




Analysis:
Here's a neutral analysis of the provided financial data summary:

**General Observations:**

*   **Data Scope:** The summary represents data for 250 periods, likely representing trading days for a single stock.
*   **Price Action:** The Open, High, Low, and Close prices indicate the trading range and closing value for each period. The mean closing price is very close to the mean opening price, suggesting that, on average, the stock closed near its opening price for the period.
*   **Trading Volume:** The volume metric shows the number of shares traded during each period. The mean volume is around 50.26 million shares.
*   **Dividends:** A small average dividend payout is observed.
*   **Stock Splits:** There have been no stock splits during this period.

**Key Insights from the Data:**

*   **Price Range:** The stock traded within a considerable range over the 250 periods. The lowest closing price was approximately 192.32, while the highest was around 285.92. This represent

In [25]:
# # Create instances of the agents with specific roles.
# data_agent = DataAgent("DataAgent", "You fetch and provide historical stock data.", client)
# analysis_agent = AnalysisAgent("AnalysisAgent", "You analyze financial data and provide neutral insights.", client)
# decision_agent = DecisionAgent("DecisionAgent", "You provide trading recommendations based on analysis.", client)

# # Step 1: Data Agent fetches historical data for a chosen stock.
# ticker = "AAPL"  # Example ticker for Apple Inc.
# data_df = data_agent.fetch_data(ticker, period='12mo', interval='1d')
# print("\nFetched Data (last 5 records):")
# print(data_df.tail())

# Step 3: Decision Agent provides a trading recommendation based on the analysis.
recommendation = decision_agent.recommend(analysis)
print("\nTrading Recommendation:")
print(recommendation)


Trading Recommendation:
**Recommendation: Hold**

**Reasoning:**

The analysis presents a neutral view of the stock's historical data. While there's been a significant price range over the 250 periods, indicating potential for both gains and losses, the current data doesn't provide a strong indication of immediate upward or downward momentum.

*   **Moderate Volatility:** The standard deviation suggests moderate price swings, which could be exploited by active traders, but for a general investor, it doesn't signal a clear buying or selling opportunity.
*   **Average Daily Range:** The average daily range is relatively small compared to the overall price range, implying that on most days, the price doesn't move drastically from its opening to closing.
*   **No Clear Trend:** The median closing price being higher than the mean suggests some positive skew, but not enough to strongly indicate an upward trend. Similarly, the absence of significant upward or downward trends is implied by th

# Conclusion

We simulated a multi-agent system using Python. The **Data Agent** retrieves historical stock data (for example, from Apple Inc. over the past month), the **Analysis Agent** computes summary statistics and generates neutral insights, and the **Decision Agent** produces a trading recommendation based on the analysis.

You can extend and refine each agent's functionality by incorporating more detailed analysis or additional financial data sources. If you have an OpenAI API key, you can obtain real-time responses from a language model instead of the simulated outputs shown here.

# Extension

## Create Report

In [28]:
# Import necessary libraries
import yfinance as yf
import pandas as pd
import datetime
import plotly.graph_objects as go
import plotly.io as pio

# Uncomment and set your API key below if available.
# openai.api_key = "your-openai-api-key"

# Function to create a Plotly line chart for the closing prices of the stock.
def create_stock_plot(data_df, ticker):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=data_df.index,
        y=data_df['Close'],
        mode='lines+markers',
        name='Close Price'
    ))
    fig.update_layout(
        title=f'{ticker} Stock Closing Prices (1 Month Daily Data)',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        template='plotly_white'
    )
    return fig

# Function to convert a DataFrame to an HTML table.
def dataframe_to_html(data_df):
    return data_df.to_html(classes="table table-striped", border=0)

def generate_html_report(data_df, analysis, recommendation, plot_fig, output_filename='financial_report.html'):
    # Convert the DataFrame to HTML.
    data_html = dataframe_to_html(data_df)
    # Convert the Plotly figure to an HTML div.
    plot_html = pio.to_html(plot_fig, full_html=False, include_plotlyjs='cdn')

    # Build the full HTML content.
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Financial Multi-Agent Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
            }}
            h1, h2 {{
                color: #333;
            }}
            .section {{
                margin-bottom: 40px;
            }}
            .table {{
                border-collapse: collapse;
                width: 100%;
            }}
            .table th, .table td {{
                border: 1px solid #ddd;
                padding: 8px;
            }}
            .table th {{
                background-color: #f2f2f2;
                text-align: left;
            }}
            pre {{
                background-color: #f8f8f8;
                padding: 10px;
                border: 1px solid #ddd;
                overflow: auto;
                white-space: pre-wrap;       /* Ensures text wraps */
                word-wrap: break-word;       /* Breaks long words */
            }}
        </style>
    </head>
    <body>
        <h1>Financial Multi-Agent Report</h1>

        <div class="section">
            <h2>Fetched Data (Last 5 Records)</h2>
            {data_html}
        </div>

        <div class="section">
            <h2>Stock Closing Prices Chart</h2>
            {plot_html}
        </div>

        <div class="section">
            <h2>Analysis</h2>
            <pre>{analysis}</pre>
        </div>

        <div class="section">
            <h2>Trading Recommendation</h2>
            <pre>{recommendation}</pre>
        </div>
    </body>
    </html>
    """

    # Write the report to an HTML file.
    with open(output_filename, 'w') as f:
        f.write(html_content)
    print(f"Report written to {output_filename}")
    # Step 6: Open the generated HTML report in the default web browser.
    import webbrowser
    webbrowser.open(output_filename)


In [31]:
# Step 2: Create a Plotly chart for the fetched data.
stock_fig = create_stock_plot(data_df, ticker)

# Step 5: Generate an HTML report combining the data, chart, analysis, and recommendation.
output_filename = 'financial_report.html'
generate_html_report(data_df, analysis, recommendation, stock_fig, output_filename=output_filename)



Report written to financial_report.html


# Exercises

## 1. Change the stock to another one

## 2. Add a new agent, which takes trends into account.
Trend-following is a general strategy which follows upward or downward trends of prices.